# Week 13 — Conduct a small model audit

**Research task:** Compare two prompt frames across hosted and local models while preserving every condition and avoiding unsupported cultural explanations.

**Python introduced:** nested loops, parameter grids, lists of records and a table created only after the plain records are understood.

This is the runnable coding component. The full literature-led chapter and slides remain to be developed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session13/session13_model_audits.ipynb)

Colab supports OpenRouter only; use local Jupyter for dual-route work.

In [ ]:
# Colab setup for the OpenRouter route.
import os as setup_os
import subprocess as setup_subprocess
from pathlib import Path as SetupPath
if SetupPath('/content').exists():
    setup_repo=SetupPath('/content/GenAI_Soc2026')
    if not setup_repo.exists(): setup_subprocess.run(['git','clone','https://github.com/cjbarrie/GenAI_Soc2026.git',str(setup_repo)],check=True)
    setup_os.chdir(setup_repo/'workbook'/'session13')
print('Working folder:',SetupPath.cwd())

## Load the course settings and SDKs

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

## Define two routes, two prompt frames and one output schema

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
routes = ["openrouter", "ollama"]
frames = {
    "individual":"Evaluate the action in terms of individual choice.",
    "collective":"Evaluate the action in terms of collective obligation.",
}
scenario = "A worker declines an unpaid request to stay late."
schema = {"type":"object","properties":{"approval":{"type":"integer","minimum":0,"maximum":100},"explanation":{"type":"string"}},"required":["approval","explanation"],"additionalProperties":False}
audit_records = []

## Run every route-by-frame condition

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
for route in routes:
    for frame_name, frame_instruction in frames.items():
        prompt = frame_instruction + " Score approval from 0 to 100. Scenario: " + scenario
        messages = [{"role":"user","content":prompt}]
        if route == "openrouter":
            with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
                response = client.chat.send(model=HOSTED_MODEL,messages=messages,temperature=0,response_format={"type":"json_schema","json_schema":{"name":"audit_response","strict":True,"schema":schema}})
            raw_output = response.choices[0].message.content
            requested_model = HOSTED_MODEL
        else:
            response = ollama.chat(model=LOCAL_MODEL,messages=messages,format=schema,options={"temperature":0})
            raw_output = response.message.content
            requested_model = LOCAL_MODEL
        parsed = json.loads(raw_output)
        record = {"route":route,"model":requested_model,"frame":frame_name,"prompt":prompt,"raw_output":raw_output,"approval":parsed["approval"]}
        audit_records.append(record)
        print("Condition record:", record)

## Inspect the plain records before making a table

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
for record in audit_records:
    print(record["route"], record["frame"], record["approval"])

## Convert the already understood records into a small table

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
import pandas as pd
audit_table = pd.DataFrame(audit_records)
print(audit_table[["route","model","frame","approval"]])

# ONE CHANGE: replace "worker" with "manager" and rerun the complete grid.

## Methodological check

The grid identifies patterned output differences under declared conditions. Route, model size, provider, prompt and training differences remain entangled; do not label the pattern a cultural or ideological cause.
## Recording

Trace one cell through both loops, make the role-word change, present one bounded observed difference and name at least two unresolved explanations.